# Kev-4B: best.pt of every paper run on Grummon 2025 and COVID (Kreps 2020)

What and why: `README.md` in this folder.

In [1]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Walk up from the notebook folder to locate the repository's `code/` root
CODE_DIR = next(
    (c for p in (Path.cwd(), *Path.cwd().parents) for c in (p, p / "code") if all((c / d).is_dir() for d in ("kev", "data", "training"))),
    None,
)
if CODE_DIR is None:
    raise RuntimeError(f"No code/ root containing kev/, data/, and training/ at or above {Path.cwd()}")

BETA_TRAINING_DIR = CODE_DIR / "train-notebooks" / "beta-training"
PAPER_DIR = BETA_TRAINING_DIR / "ms_paper_beta_loss"

# Run from `code/` so data loaders and shared disk caches resolve identically to the training notebooks
os.chdir(CODE_DIR)
for _p in (str(CODE_DIR), str(CODE_DIR / "kev"), str(BETA_TRAINING_DIR), str(PAPER_DIR)):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from models.kev_pointer import KevPointerModel
from training import free_gpu

import beta_ols
from beta_train import runs as run_store
from beta_train.model_state import reset_trainable, set_checkpointing, snapshot_trainable
from beta_train.progress import PROGRESS_INTERVAL_SEC, HtmlProgress
from beta_train.validation import PopulationValidator, validate_all, validate_all_cached, validation_metrics_frame

NOTEBOOK = "04-test-best-4b"
CACHE_DIR = CODE_DIR / ".cache" / "beta_training"
MODEL_SLUG = "kev-4b"
# Runs pulled from Rorqual by pull-rorqual-runs.sh (run files + checkpoints/best.pt)
RUNS_DIR = PAPER_DIR / "runs-rorqual" / MODEL_SLUG
# Each run's test results go in <run folder>/test_best/
TEST_FOLDER_NAME = "test_best"
ROW1_DIR = RUNS_DIR / "row1_base_untrained"

print(f"RUNS_DIR: {RUNS_DIR}")


RUNS_DIR: /home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/runs-rorqual/kev-4b


In [2]:
# Same backbone and LoRA shape as the training notebooks, so best.pt loads into it
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required")
device = "cuda"
free_gpu()

KEV_BASE = "jaredpalmer/kev-4b@6cfce5c2fa4b4bd64026336ab649c5ca78857d52"
LORA_RANK = 64
WEIGHTS_DTYPE = "bf16"
HEAD_TEMPERATURE = 1.0

kev_model = KevPointerModel.build(
    KEV_BASE,
    device=device,
    lora_rank=LORA_RANK,
    weights_dtype=WEIGHTS_DTYPE,
)
kev_model.net.head.temperature = HEAD_TEMPERATURE
set_checkpointing(kev_model, False)

INIT_TRAINABLE_STATE = snapshot_trainable(kev_model)
# Same key as the training notebooks, so their cached step-0 validations are reused
MODEL_KEY = {
    "base": KEV_BASE,
    "lora_rank": LORA_RANK,
    "weights_dtype": WEIGHTS_DTYPE,
    "head_temperature": HEAD_TEMPERATURE,
}
print(f"{KEV_BASE}: {len(INIT_TRAINABLE_STATE)} trainable tensors")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

jaredpalmer/kev-4b@6cfce5c2fa4b4bd64026336ab649c5ca78857d52: 500 trainable tensors


## Test datasets

In [3]:
TEST_DATASETS = ["grummon2025", "covid"]
VALIDATION_MAX_QUESTIONS = 1024
VALIDATION_MAX_CHARS = 1_000_000

validators = {}
for dataset_name in TEST_DATASETS:
    dataset_module = beta_ols.DATASET_MODULES[dataset_name]
    choice_df = dataset_module.load_choice_df()
    rows = dataset_module.regression_rows(choice_df)
    specs = dataset_module.regression_specs(rows)
    validators[dataset_name] = PopulationValidator(
        dataset_module,
        choice_df,
        specs,
        max_questions=VALIDATION_MAX_QUESTIONS,
        max_chars=VALIDATION_MAX_CHARS,
    )
display(pd.DataFrame([v.describe() for v in validators.values()]).set_index("validator"))


# Writes metrics.csv (one row per test dataset), betas.csv (model vs human betas, stacked) and test.json (what was tested)
def save_test_results(test_dir: Path, validation: dict, test_info: dict) -> None:
    test_dir.mkdir(parents=True, exist_ok=True)
    metrics_df = validation_metrics_frame(validation).T
    metrics_df.index.name = "dataset"
    run_store.write_text_atomic(test_dir / "metrics.csv", metrics_df.to_csv())
    run_store.write_betas_file(test_dir / "betas.csv", validation)
    run_store.write_text_atomic(test_dir / "test.json", json.dumps(test_info, indent=2))


# True when test_dir already holds results for this checkpoint step and these test datasets
def already_tested(test_dir: Path, checkpoint_step: int) -> bool:
    test_json = test_dir / "test.json"
    if not test_json.exists():
        return False
    stored = json.loads(test_json.read_text())
    same_step = stored["checkpoint_step"] == checkpoint_step
    same_datasets = stored["test_datasets"] == TEST_DATASETS
    return same_step and same_datasets


,kind,tasks,packed_persona_states,regressions,max_questions,max_chars,graph_limits,prep_sec
validator,,,,,,,,
grummon2025,fused,33798,4906,grummon2025,1024,1000000,"BANK_WIDTH=512, GRAPH_STATE=512, GRAPH_STATES=...",6.502
covid,fused,9855,1845,covid,1024,1000000,"BANK_WIDTH=512, GRAPH_STATE=512, GRAPH_STATES=...",1.817


## Row 1: untrained model

In [5]:
ROW1_DIR / TEST_FOLDER_NAME

PosixPath('/home/dw/github/decision-models-finetuning/code/train-notebooks/beta-training/ms_paper_beta_loss/runs-rorqual/kev-4b/row1_base_untrained/test_best')

In [6]:
# Untrained model (step 0), cached in code/.cache/beta_training
step0_validation, step0_cache_report = validate_all_cached(
    kev_model,
    validators,
    INIT_TRAINABLE_STATE,
    model_key=MODEL_KEY,
    cache_dir=CACHE_DIR,
)
display(step0_cache_report)
step0_info = {
    "notebook": NOTEBOOK,
    "model": MODEL_KEY,
    "row": "Row 1",
    "checkpoint": "untrained",
    "checkpoint_step": 0,
    "test_datasets": TEST_DATASETS,
}
save_test_results(ROW1_DIR / TEST_FOLDER_NAME, step0_validation, step0_info)
display(validation_metrics_frame(step0_validation).round(5))


,source
validator,
grummon2025,computed
covid,computed


,grummon2025,covid
kl_loss,1.09993,1.52537
beta_mse,0.02110,0.00908
beta_mae,0.12328,0.07255
beta_mape_pct,146.12472,123.51566
spearman_rho,0.80000,0.87273
eval_prep_sec,0.55120,2.75562
eval_gpu_sec,308.78822,137.69905
eval_wall_sec,314.09974,143.41663
eval_ms_per_task,9.29344,14.55268


## best.pt of every finished run

In [7]:
# best.pt of every finished run in RUNS_DIR; unfinished runs are skipped (rerun this notebook once they finish)
progress_bar = HtmlProgress(
    total=0, initial=0, description="test (waiting)", unit="batch", min_interval_sec=PROGRESS_INTERVAL_SEC
)
for run_dir in sorted(RUNS_DIR.iterdir()):
    if not (run_dir / "run.json").exists():
        continue
    run_info = run_store.read_run_json(run_dir)
    row_code = run_info["identity"]["run_metadata"]["paper_row"]
    if run_info["status"] != "finished":
        print(f"{row_code} {run_dir.name}: {run_info['status']}, skipped")
        continue
    test_dir = run_dir / TEST_FOLDER_NAME
    checkpoint = run_store.load_best(run_dir)
    best = checkpoint["best"]
    checkpoint_step = int(best["step"])
    if already_tested(test_dir, checkpoint_step):
        print(f"{row_code} {run_dir.name}: best.pt (step {checkpoint_step}) already tested, skipped")
        continue
    reset_trainable(kev_model, checkpoint["trainable"])
    validation = validate_all(kev_model, validators, progress_bar)
    test_info = {
        "notebook": NOTEBOOK,
        "model": MODEL_KEY,
        "row": row_code,
        "checkpoint": "best.pt",
        "checkpoint_step": checkpoint_step,
        "best_metric": best["metric"],
        "best_value": float(best["value"]),
        "test_datasets": TEST_DATASETS,
    }
    save_test_results(test_dir, validation, test_info)
    print(f"{row_code} {run_dir.name}: best.pt (step {checkpoint_step}) tested")
    display(validation_metrics_frame(validation).round(5))
progress_bar.close()
reset_trainable(kev_model, INIT_TRAINABLE_STATE)


Row 2 (Ablation) row2_ablation_wu_table_no_kl: best.pt (step 96) tested


,grummon2025,covid
kl_loss,0.93407,1.46188
beta_mse,0.01549,0.00697
beta_mae,0.10288,0.06481
beta_mape_pct,121.96541,113.07271
spearman_rho,0.80000,0.92727
eval_prep_sec,0.02547,0.00873
eval_gpu_sec,308.94058,137.63936
eval_wall_sec,311.19563,139.72005
eval_ms_per_task,9.20752,14.17758


Row 2 row2_wu_table_anchored: best.pt (step 320) tested


,grummon2025,covid
kl_loss,0.87123,1.20178
beta_mse,0.00633,0.00591
beta_mae,0.07028,0.05736
beta_mape_pct,78.53347,104.56280
spearman_rho,0.80000,0.92727
eval_prep_sec,0.02500,0.00866
eval_gpu_sec,308.87448,137.58313
eval_wall_sec,311.03033,139.65739
eval_ms_per_task,9.20263,14.17122


Row 2b row2b_wu_ce_oracle: best.pt (step 416) tested


,grummon2025,covid
kl_loss,0.88710,1.11967
beta_mse,0.00357,0.00173
beta_mae,0.05164,0.03364
beta_mape_pct,61.83906,60.40597
spearman_rho,0.80000,0.94545
eval_prep_sec,0.02497,0.00866
eval_gpu_sec,308.78486,137.61781
eval_wall_sec,310.92357,139.64131
eval_ms_per_task,9.19947,14.16959


Row 3 row3_berke_ce_only: best.pt (step 544) tested


,grummon2025,covid
kl_loss,0.95915,1.19461
beta_mse,0.00704,0.00463
beta_mae,0.06520,0.05738
beta_mape_pct,80.56121,101.93680
spearman_rho,0.80000,0.93636
eval_prep_sec,0.02498,0.00850
eval_gpu_sec,308.82791,137.63732
eval_wall_sec,310.95472,139.66123
eval_ms_per_task,9.20039,14.17161


Row 4 row4_hybrid_berke_ce_wu_table_anchored: running, skipped
Row 4 (Ablation) row4_hybrid_berke_ce_wu_table_no_kl: running, skipped
Row 5 row5_full_berke_wu_ce_oracle: running, skipped


## Summary

In [8]:
# Every tested checkpoint of this model: one row per (row, run folder, test dataset)
summary_frames = []
for test_json in sorted(RUNS_DIR.glob(f"*/{TEST_FOLDER_NAME}/test.json")):
    test_dir = test_json.parent
    test_info = json.loads(test_json.read_text())
    metrics_df = pd.read_csv(test_dir / "metrics.csv")
    metrics_df.insert(0, "checkpoint_step", test_info["checkpoint_step"])
    metrics_df.insert(0, "slug", test_dir.parent.name)
    metrics_df.insert(0, "row", test_info["row"])
    summary_frames.append(metrics_df)
summary_df = pd.concat(summary_frames, ignore_index=True)
summary_df = summary_df.sort_values(["dataset", "row", "slug"]).set_index(["dataset", "row", "slug"])
display(summary_df[["checkpoint_step", "beta_mse", "beta_mae", "spearman_rho", "kl_loss"]].round(5))


checkpoint_step  beta_mse  beta_mae  spearman_rho  kl_loss
dataset     row              slug                                                                                    
covid       Row 1            row1_base_untrained                         0   0.00908   0.07255       0.87273  1.52537
            Row 2            row2_wu_table_anchored                    320   0.00591   0.05736       0.92727  1.20178
            Row 2 (Ablation) row2_ablation_wu_table_no_kl               96   0.00697   0.06481       0.92727  1.46188
            Row 2b           row2b_wu_ce_oracle                        416   0.00173   0.03364       0.94545  1.11967
            Row 3            row3_berke_ce_only                        544   0.00463   0.05738       0.93636  1.19461
grummon2025 Row 1            row1_base_untrained                         0   0.02110   0.12328       0.80000  1.09993
            Row 2            row2_wu_table_anchored                    320   0.00633   0.07028       0.80000  0.87123
            Row 2 (Ablation) row2_ablation_wu_table_no_kl               96   0.01549   0.10288       0.80000  0.93407
            Row 2b           row2b_wu_ce_oracle                        416   0.00357   0.05164       0.80000  0.88710
            Row 3            row3_berke_ce_only                        544   0.00704   0.06520       0.80000  0.95915